In [ ]:
#!/usr/bin/env python3
"""
MOFA(+)-style regression TEST pipeline (Python).

Assumes train_mofa.R saves:
  - ../models/tea/<target>_regression/<split_tag>_mofa_model.rds
  - ../models/tea/<target>_regression/<split_tag>_mofa_model.hdf5
  - ../models/tea/<target>_regression/<split_tag>_<target>_linear_reg.pkl

The HDF5 export is required here since Python cannot consume the .rds MOFA
object directly.

TEST does:
  - load FULL data (rna/atac/adt_minus_target)
  - subset to TEST via split indices
  - load MOFA HDF5 weights W_by_view (RNA/ATAC/ADT)
  - compute Z_test via the same joint ridge projection used in train_mofa.R
  - load sklearn LinearRegression regressor and predict
  - save metrics + predictions under ../results/tea/<split_tag>_mofa_reg_<target>/
"""

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp

import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score




In [ ]:
# -----------------------------
# 1) Data loaders (match R)
# -----------------------------

def load_tea_seq(base_dir: str = "../data", target_protein: str = "CD45RA"):
    base = Path(base_dir)

    rna = ad.read_h5ad(base / "rna.h5ad")
    atac = ad.read_h5ad(base / "atac.h5ad")
    adt = ad.read_h5ad(base / f"adt_minus_{target_protein}.h5ad")

    if not np.array_equal(rna.obs_names.astype(str), atac.obs_names.astype(str)):
        raise ValueError("RNA and ATAC obs_names are not aligned.")
    if not np.array_equal(rna.obs_names.astype(str), adt.obs_names.astype(str)):
        raise ValueError("RNA and ADT obs_names are not aligned.")

    return rna, atac, adt

def load_test_indices(splits_dir: str, split_tag: str, n_cells: int) -> np.ndarray:
    """
    Match the R helper read_index_csv_0based():

    - Read the first numeric column.
    - If it exactly spans 0..n_cells-1 -> treat as 0-based.
    - Else if it exactly spans 1..n_cells -> convert to 0-based.
    - Else default to 0-based (no guessing shift).
    - Validate in-range and no silent clipping.
    """
    path = Path(splits_dir) / f"{split_tag}_test_idx.csv"
    df = pd.read_csv(path, header=0)

    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        raise ValueError(f"No numeric column found in {path}")

    idx = df[num_cols[0]].to_numpy()

    # check NaNs
    if np.isnan(idx).any():
        bad = np.where(np.isnan(idx))[0][:10]
        raise ValueError(f"NaNs in {path} at rows {bad.tolist()}.")

    # require integer-valued
    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        bad = np.where(~np.isclose(idx, idx_int))[0][:10]
        raise ValueError(f"Non-integer indices in {path} at rows {bad.tolist()}: {idx[bad].tolist()}")

    idx = idx_int
    mn, mx = int(idx.min()), int(idx.max())

    # R-style detection
    if mn == 0 and mx == (n_cells - 1):
        idx0 = idx
        base = "0-based (exact sentinel match)"
    elif mn == 1 and mx == n_cells:
        idx0 = idx - 1
        base = "1-based (exact sentinel match) -> converted"
    else:
        idx0 = idx
        base = "0-based (default)"

    # strict range check (no clipping)
    if (idx0 < 0).any() or (idx0 >= n_cells).any():
        bad = np.where((idx0 < 0) | (idx0 >= n_cells))[0][:10]
        raise ValueError(
            f"Out-of-range indices after base handling in {path}. "
            f"min={idx0.min()}, max={idx0.max()}, n_cells={n_cells}. "
            f"Example bad positions {bad.tolist()} with values {idx0[bad].tolist()}."
        )

    print(f"[load_test_indices] {path.name}: {len(idx0)} indices, {base}, range=[{idx0.min()},{idx0.max()}]")
    return idx0.astype(np.int64)


def load_regression_target(target_protein: str, base_dir: str, cell_names: pd.Index) -> np.ndarray:
    path = Path(base_dir) / "response" / f"{target_protein}.csv"
    df_y = pd.read_csv(path, index_col=0)
    return df_y.loc[cell_names].iloc[:, 0].values.astype(np.float32)

def get_norm(adata: ad.AnnData, layer: str = "norm"):
    X = adata.layers[layer] if layer in adata.layers else adata.X
    return X.tocsr() if sp.issparse(X) else np.asarray(X)




In [ ]:
# -----------------------------
# 2) Same joint ridge projection as R
# -----------------------------

def joint_ridge_project(
    X_by_view: dict[str, np.ndarray],
    W_by_view: dict[str, np.ndarray],
    lam: float = 1e-3,
) -> np.ndarray:
    """
    R code:
      A <- lam * diag(K); B <- 0 (n_samp x K)
      for v:
        B <- B + t(X) %*% W
        A <- A + t(W) %*% W
      Z <- B %*% solve(A)

    Here:
      X_by_view[v] is (features x n_cells)
      W_by_view[v] is (features x K)
    Returns:
      Z (n_cells x K)
    """
    views = list(X_by_view.keys())
    if not views:
        raise ValueError("Empty X_by_view")

    K = W_by_view[views[0]].shape[1]
    n = X_by_view[views[0]].shape[1]

    A = lam * np.eye(K, dtype=np.float64)
    B = np.zeros((n, K), dtype=np.float64)

    for v in views:
        X = X_by_view[v]
        W = W_by_view[v]
        if X.shape[0] != W.shape[0]:
            raise ValueError(f"[{v}] Feature mismatch: X has {X.shape[0]} rows, W has {W.shape[0]} rows")
        if X.shape[1] != n:
            raise ValueError(f"[{v}] Cell mismatch: X has {X.shape[1]} cols, expected {n}")
        if W.shape[1] != K:
            raise ValueError(f"[{v}] Factor mismatch: W has {W.shape[1]} cols, expected {K}")

        B += (X.T @ W)
        A += (W.T @ W)

    Z = B @ np.linalg.inv(A)
    return Z.astype(np.float32)




In [ ]:
# -----------------------------
# 3) Metrics
# -----------------------------

def r2_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_bar = y_true.mean()

    ss_tot = np.sum((y_true - y_bar)**2)
    ss_res = np.sum((y_true - y_pred)**2)

    return 1.0 - ss_res / ss_tot


def evaluate_regression(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    eps = 1e-12

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_manual(y_true, y_pred))
    evs = float(explained_variance_score(y_true, y_pred))

    yt = (y_true - y_true.mean()) / (y_true.std(ddof=0) + eps)
    yp = (y_pred - y_pred.mean()) / (y_pred.std(ddof=0) + eps)
    pearson_r = float(np.clip((yt * yp).mean(), -1.0, 1.0))

    try:
        from scipy.stats import spearmanr
        spearman_r = float(spearmanr(y_true, y_pred).correlation)
    except Exception:
        spearman_r = float("nan")

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "explained_var": evs,
        "pearson_r": pearson_r,
        "spearman_r": spearman_r,
    }




In [ ]:
# -----------------------------
# 4) Load MOFA weights from HDF5 (MOFA2 export)
# -----------------------------

def load_mofa_weights_from_hdf5(mofa_hdf5: str | Path) -> dict[str, np.ndarray]:
    """
    Loads W weights per view from a MOFA2 HDF5 export.

    Returns:
      dict view_name -> W (features x K) float32

    Notes:
      MOFA2 internal layout differs across versions. We search for an "expectations/W" group.
    """
    import h5py

    mofa_hdf5 = str(mofa_hdf5)
    with h5py.File(mofa_hdf5, "r") as f:

        # find candidate group paths ending with '/expectations/W' or containing it
        def iter_groups(g, prefix=""):
            for k, obj in g.items():
                p = f"{prefix}/{k}"
                if isinstance(obj, h5py.Group):
                    yield p, obj
                    yield from iter_groups(obj, p)

        cand = []
        for p, g in iter_groups(f, ""):
            if p.lower().endswith("/expectations/w") or ("/expectations/" in p.lower() and p.lower().endswith("/w")):
                cand.append((p, g))

        if not cand:
            raise RuntimeError(
                f"Could not locate MOFA weights group 'expectations/W' in: {mofa_hdf5}.\n"
                "Inspect the HDF5 structure (e.g., with h5ls) and adjust loader."
            )

        # pick shortest path as best guess
        cand.sort(key=lambda x: len(x[0]))
        W_group_path, W_group = cand[0]

        W_by_view: dict[str, np.ndarray] = {}
        for view_name, obj in W_group.items():
            if hasattr(obj, "shape"):  # Dataset
                W = np.array(obj, dtype=np.float32)
            else:
                # nested group, try common datasets
                found = None
                for k in ("value", "mean", "data", "W"):
                    if k in obj:
                        found = obj[k]
                        break
                if found is None:
                    continue
                W = np.array(found, dtype=np.float32)

            # heuristic transpose: many exports store (K x features)
            if W.shape[0] < W.shape[1]:
                W = W.T
            W_by_view[view_name] = W

        if not W_by_view:
            raise RuntimeError(f"Found {W_group_path} but extracted no per-view W datasets.")

    return W_by_view




In [ ]:
# -----------------------------
# 5) TEST pipeline
# -----------------------------

def test_mofa_regression(
    target_protein: str = "CD45RA",
    dataset_name: str = "tea",
    split_tag: str = "tea_split3_all_celltypes",
    base_dir: str = "../data",
    splits_dir: str = "../splits",
    model_base: str = "../models/tea",
    results_base: str = "../results/tea",
    lam: float = 1e-3,
    save_Z: bool = True,
) -> None:
    base_dir = Path(base_dir).resolve()
    splits_dir = Path(splits_dir).resolve()
    model_base = Path(model_base).resolve()
    results_base = Path(results_base).resolve()

    # match your R dir structure
    model_dir = model_base / f"{target_protein}_regression"
    mofa_hdf5 = model_dir / f"{split_tag}_mofa_model.hdf5"
    reg_path = model_dir / f"{split_tag}_{target_protein}_linear_reg.pkl"

    if not mofa_hdf5.exists():
        raise FileNotFoundError(
            f"Missing: {mofa_hdf5}\n"
            "Your R script currently only *prints* the intended HDF5 path.\n"
            "Fix in R: MOFA2::save_model(mofa_trained, file = mofa_hdf5_path)"
        )
    if not reg_path.exists():
        raise FileNotFoundError(f"Missing regressor: {reg_path}")

    out_root = results_base / f"{split_tag}_mofa_reg_{target_protein}"
    out_root.mkdir(parents=True, exist_ok=True)

    # load full data, subset test
    rna, atac, adt = load_tea_seq(base_dir=str(base_dir), target_protein=target_protein)
    test_idx = load_test_indices(str(splits_dir), split_tag, rna.n_obs)
    test_cells = rna.obs_names[test_idx]
    print("PY first 5 obs:", rna.obs_names[:5].tolist())
    print("PY last 5 obs:", rna.obs_names[-5:].tolist())
    print(f"[MOFA-TEST] #TEST cells: {len(test_idx)}")


    rna_te = rna[test_cells].copy()
    atac_te = atac[test_cells].copy()
    adt_te = adt[test_cells].copy()

    y_true = load_regression_target(target_protein, str(base_dir), test_cells)

    # load MOFA weights
    print(f"[MOFA] Loading W from {mofa_hdf5}")
    W_by_view = load_mofa_weights_from_hdf5(mofa_hdf5)

    # Build X_by_view as (features x cells), matching the R code t(py_to_dgC(...))
    # We keep the naming consistent with your R views: RNA / ATAC / ADT
    def X_feat_by_cell(A: ad.AnnData) -> np.ndarray:
        X = get_norm(A, layer="norm")
        # keep as dense for matmul; if too big you need a sparse-safe implementation
        if sp.issparse(X):
            X = X.toarray()
        return X.T.astype(np.float32)

    X_by_view = {
        "RNA": X_feat_by_cell(rna_te),
        "ATAC": X_feat_by_cell(atac_te),
        "ADT": X_feat_by_cell(adt_te),
    }

    # Minimal dimension sanity checks (robust alignment requires feature names from MOFA export)
    for v in ("RNA", "ATAC", "ADT"):
        if v not in W_by_view:
            raise KeyError(f"View '{v}' missing in W. Found: {list(W_by_view.keys())}")
        if X_by_view[v].shape[0] != W_by_view[v].shape[0]:
            raise ValueError(
                f"[Feature mismatch] view={v}: X has {X_by_view[v].shape[0]} features, "
                f"W has {W_by_view[v].shape[0]}.\n"
                "This indicates the MOFA model was trained on a different feature set/order.\n"
                "Robust fix: save feature names per view from training and reindex TEST accordingly."
            )

    # Project test into factors
    Z_test = joint_ridge_project(X_by_view=X_by_view, W_by_view=W_by_view, lam=lam)
    if save_Z:
        np.save(out_root / "Z_test.npy", Z_test)

    # Predict
    reg = joblib.load(reg_path)
    y_pred = reg.predict(Z_test).astype(np.float32)

    # Evaluate
    report = evaluate_regression(y_true, y_pred)
    print("[TEST Metrics]", report)

    # Save
    np.save(out_root / "y_true.npy", y_true)
    np.save(out_root / "y_pred.npy", y_pred)
    np.save(out_root / "idx_test.npy", test_idx.astype(int))

    pd.DataFrame({"cell": test_cells.astype(str), "y_true": y_true, "y_pred": y_pred}).to_csv(
        out_root / "test_predictions.csv", index=False
    )

    report_json = dict(report)
    report_json["run"] = {
        "method": "MOFA (W from HDF5) + joint_ridge_project + sklearn.LinearRegression",
        "task": "regression",
        "target": target_protein,
        "dataset": dataset_name,
        "split_tag": split_tag,
        "lam": float(lam),
        "mofa_hdf5": str(mofa_hdf5),
        "regressor": str(reg_path),
        "n_test": int(len(test_idx)),
    }
    report_json["timestamp"] = datetime.now().isoformat(timespec="seconds")

    with open(out_root / "test_metrics.json", "w") as f:
        json.dump(report_json, f, indent=2)

    print("[Saved] outputs under:", out_root)


if __name__ == "__main__":
    test_mofa_regression(target_protein="CD45RA")
